In [1]:
#!pip install transformers datasets torch evaluate rouge_score


In [6]:
import wandb
wandb.init(mode="disabled", project="qa_squad_demo")
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, pipeline
from datasets import load_dataset
import torch
import math

# 1. Load a text dataset (e.g., `imdb` for conversational-like data)
dataset = load_dataset("imdb")
small_train = dataset["train"].select(range(1000))
small_validation = dataset["test"].select(range(200))

# 2. Load the tokenizer and model
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Ensure the tokenizer can handle padding
tokenizer.pad_token = tokenizer.eos_token

def preprocess_function(examples):
    inputs = examples["text"]

    # Tokenize inputs
    tokenized = tokenizer(
        inputs,
        max_length=128,
        truncation=True,
        padding="max_length",
        return_tensors="pt"
    )

    tokenized["labels"] = tokenized["input_ids"].clone()

    return tokenized

# Process datasets
tokenized_train = small_train.map(
    preprocess_function,
    batched=True,
    remove_columns=small_train.column_names,
    batch_size=8
)

tokenized_validation = small_validation.map(
    preprocess_function,
    batched=True,
    remove_columns=small_validation.column_names,
    batch_size=8
)

# Training arguments
training_args = TrainingArguments(
    output_dir="./results",               # 模型儲存資料夾
    num_train_epochs=2,                   # 訓練輪數
    per_device_train_batch_size=8,        # 訓練用 batch size
    per_device_eval_batch_size=8,         # 驗證用 batch size
    learning_rate=5e-5,                   # 學習率
    weight_decay=0.01,                    # L2 正則化
    logging_dir="./logs",                 # 日誌資料夾
    logging_steps=50                      # 每 50 步記錄一次
)

# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation
)

# Train the model
trainer.train()

# Save the model
model.save_pretrained("./text_gen_model")
tokenizer.save_pretrained("./text_gen_model")


Map:   0%|          | 0/200 [00:00<?, ? examples/s]

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
50,3.731700
100,3.616600
150,3.513100
200,3.403500
250,3.368000


('./text_gen_model\\tokenizer_config.json',
 './text_gen_model\\special_tokens_map.json',
 './text_gen_model\\vocab.json',
 './text_gen_model\\merges.txt',
 './text_gen_model\\added_tokens.json',
 './text_gen_model\\tokenizer.json')

In [7]:
# Test the model with a text generation pipeline
text_gen_pipeline = pipeline("text-generation", model="./text_gen_model", tokenizer="./text_gen_model")

# Example: Chatbot-like behavior
context = "User: How are you?\nBot: I am"
prediction = text_gen_pipeline(context, max_length=50, num_return_sequences=1, pad_token_id=tokenizer.eos_token_id)

print(f"\nInput: {context}")
print(f"Response: {prediction[0]['generated_text']}")

# 1. Perplexity Evaluation
print("\n--- Perplexity Evaluation ---")
eval_results = trainer.evaluate()
print(f"Evaluation Loss: {eval_results['eval_loss']}")
print(f"Perplexity: {math.exp(eval_results['eval_loss'])}")

# 2. Qualitative Evaluation
test_contexts = [
    "User: Tell me about AI.\nBot:",
    "User: What's the weather like?\nBot:",
    "User: Recommend me a movie.\nBot:"
]

print("\n--- Qualitative Evaluation ---")
for context in test_contexts:
    prediction = text_gen_pipeline(context, max_length=50, num_return_sequences=1, pad_token_id=tokenizer.eos_token_id)
    print(f"\nInput: {context}")
    print(f"Response: {prediction[0]['generated_text']}")
import evaluate

# 3. BLEU/ROUGE Scores
print("\n--- BLEU/ROUGE Evaluation ---")
import evaluate
metric = evaluate.load("rouge")
# Generate predictions
predictions = [
    text_gen_pipeline(context, max_length=50, num_return_sequences=1, pad_token_id=tokenizer.eos_token_id)[0]["generated_text"]
    for context in test_contexts
]

# Reference responses
references = [
    "Bot: Artificial Intelligence (AI) refers to the simulation of human intelligence in machines.",
    "Bot: I don't know the weather right now, but you can check your local forecast!",
    "Bot: I recommend you watch 'Inception,' a thrilling movie with a unique plot."
]

results = metric.compute(predictions=predictions, references=references, use_stemmer=True)
print(results)




Device set to use cuda:0
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Input: User: How are you?
Bot: I am
Response: User: How are you?
Bot: I am sorry dude. I am so sorry. <br /><br />So the first thing we do is put a special order for all our films. We have a special order for the film "Wanna Come by?" and we get it in the mail. We get it in a box and send it to the store. We then check on it and we check on it every day. The movies are made and we don't know how to get it in the mail.<br /><br />The second thing we do is the special order for the film "My Little Pony: Friendship Is Magic." We get it in the mail and we check it and we say we have it in the mail. And we get it in the mail. And we wait. And then we check on it and we wait. And we wait. And then we wait. And we wait. And we wait. And we wait.<br /><br />The third thing we do is we go to the store and buy it the same day.<br /><br />And then we wait in line at the store for about 10 seconds before we get the movie.<br /><br />And then we just sit there and wait for it to come out. Then, we

Evaluation Loss: 3.5388126373291016
Perplexity: 34.426018744266266

--- Qualitative Evaluation ---


Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Input: User: Tell me about AI.
Bot:
Response: User: Tell me about AI.
Bot: Yeah, I know, really. I just saw this movie. But I'll bet that you can hardly believe what you're reading. I believe it to be a horror movie. I would not have rated it under the "worst" category. I thought it was a horror movie. But the acting was so bad that I thought it was a horror movie. I also thought the aliens were big and scary, and that the aliens was so strong that it made me wonder if I could ever be a hero. I thought that was the movie that I had heard about. But it was like that movie made me laugh out loud. You know, the one where you see a little girl with a bunch of kids. And she is trying to get out of her car. And she gets stuck on a cliff. And it makes her look like she is about to die. It even makes her say "Oh my God!" It was like that movie made you laugh out loud. But I guess the movie you're watching is really just a horror movie. I don't know what the acting is, but I think it's pretty 

...


Input: User: What's the weather like?
Bot:
Response: User: What's the weather like?
Bot: <br /><br />There's no rain so far. If you were to say, "Ah, wait! I'm back!" no one would have guessed it from the above. <br /><br />I didn't know it was so bad. I'm a very big fan of the series and it was just me and a friend that had it figured. I was a bit disappointed to learn that the characters were actually in the series, which was kind of disappointing. But it was a good movie, and I'll admit it was pretty good. I just can't wait for the next one.<br /><br />I liked the plot, but I can't wait for the next one.<br /><br />I hate the idea that the story could have been something more. That can't be true. The character development is great, but the writing for the characters is so bad, it's hard to believe that it could have been written by a writer like me. I wish I could have written this book for a girl at the age of 10, but that would have been too much for me. <br /><br />The story is 

Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Input: User: Recommend me a movie.
Bot:
Response: User: Recommend me a movie.
Bot:

--- BLEU/ROUGE Evaluation ---


Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'rouge1': 0.07008670968061831, 'rouge2': 0.012170940170940172, 'rougeL': 0.057085422567656065, 'rougeLsum': 0.06670261492596517}
